# Notebook 3: Churn Prediction Model
**Project:** Customer Churn Intelligence System — Telecom  
**Author:** Prathmesh Joshi

---
### Goal
Build a logistic regression model that assigns a **churn probability score** to every customer.  
This score feeds into the Power BI dashboard as a risk tier (High / Medium / Low).

## Step 1: Import Libraries & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix,
                              roc_curve, classification_report)

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

df = pd.read_csv('../data/telco_churn_clean.csv')
print(f'Loaded: {df.shape[0]:,} rows, {df.shape[1]} columns')

## Step 2: Prepare Features for Modeling

In [ ]:
# One-hot encode remaining categorical columns
df_model = pd.get_dummies(df, columns=['InternetService', 'Contract', 'PaymentMethod'],
                           drop_first=False)

# Drop columns not useful for modeling
drop_cols = ['customerID', 'tenure_bucket']
df_model = df_model.drop(columns=drop_cols, errors='ignore')

# Make sure all columns are numeric
df_model = df_model.select_dtypes(include=[np.number])

print('Feature columns for model:')
print(df_model.columns.tolist())
print(f'\nTotal features: {df_model.shape[1] - 1} (excluding target)')

## Step 3: Train / Test Split

In [ ]:
X = df_model.drop('Churn', axis=1)
y = df_model['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set: {X_train.shape[0]:,} customers')
print(f'Test set:     {X_test.shape[0]:,} customers')
print(f'\nChurn rate in train: {y_train.mean():.1%}')
print(f'Churn rate in test:  {y_test.mean():.1%}')

## Step 4: Scale Features & Train Model

In [ ]:
# Scale features (important for logistic regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Train logistic regression model
model = LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced')
model.fit(X_train_scaled, y_train)

print('Model trained successfully.')

## Step 5: Evaluate Model Performance

In [ ]:
y_pred       = model.predict(X_test_scaled)
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]

print('=' * 45)
print('        MODEL PERFORMANCE REPORT')
print('=' * 45)
print(f' Accuracy:   {accuracy_score(y_test, y_pred):.1%}')
print(f' Precision:  {precision_score(y_test, y_pred):.1%}')
print(f' Recall:     {recall_score(y_test, y_pred):.1%}')
print(f' F1 Score:   {f1_score(y_test, y_pred):.1%}')
print(f' ROC-AUC:    {roc_auc_score(y_test, y_pred_proba):.1%}')
print('=' * 45)
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['Stayed', 'Churned']))

## Step 6: Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicted: Stayed', 'Predicted: Churned'],
            yticklabels=['Actual: Stayed', 'Actual: Churned'],
            linewidths=1, linecolor='white', ax=ax)
ax.set_title('Confusion Matrix — Churn Prediction Model', fontsize=13, pad=12)

plt.tight_layout()
plt.savefig('../images/07_confusion_matrix.png', bbox_inches='tight')
plt.show()
print('Chart saved: images/07_confusion_matrix.png')

## Step 7: ROC Curve

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
auc_score   = roc_auc_score(y_test, y_pred_proba)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(fpr, tpr, color='#DD8452', lw=2.5, label=f'ROC Curve (AUC = {auc_score:.2f})')
ax.plot([0, 1], [0, 1], color='gray', lw=1.5, linestyle='--', label='Random Classifier')
ax.fill_between(fpr, tpr, alpha=0.1, color='#DD8452')
ax.set_title('ROC Curve — Churn Prediction Model', fontsize=13, pad=12)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.legend(fontsize=11)

plt.tight_layout()
plt.savefig('../images/08_roc_curve.png', bbox_inches='tight')
plt.show()
print('Chart saved: images/08_roc_curve.png')

## Step 8: Score All Customers & Assign Risk Tiers

In [ ]:
# Score every customer in the full dataset
X_all_scaled = scaler.transform(X)
churn_scores = model.predict_proba(X_all_scaled)[:, 1]

# Add scores back to original dataframe
df_orig = pd.read_csv('../data/telco_churn_clean.csv')
df_orig['churn_probability'] = (churn_scores * 100).round(1)

# Assign risk tiers
def risk_tier(score):
    if score >= 65:
        return 'High Risk'
    elif score >= 35:
        return 'Medium Risk'
    else:
        return 'Low Risk'

df_orig['risk_tier'] = df_orig['churn_probability'].apply(risk_tier)

print('Risk tier distribution:')
print(df_orig['risk_tier'].value_counts().to_string())

print(f'\nHigh-risk customers:  {(df_orig["risk_tier"]=="High Risk").sum():,}')
print(f'Revenue at risk:      ${df_orig[df_orig["risk_tier"]=="High Risk"]["MonthlyCharges"].sum():,.0f}/month')

## Step 9: Top Feature Importances

In [ ]:
feature_names = X.columns.tolist()
coefficients  = model.coef_[0]

feat_importance = pd.DataFrame({
    'feature': feature_names,
    'coefficient': coefficients,
    'abs_coeff': np.abs(coefficients)
}).sort_values('abs_coeff', ascending=False).head(12)

fig, ax = plt.subplots(figsize=(9, 6))
colors = ['#DD8452' if c > 0 else '#4C72B0' for c in feat_importance['coefficient']]
bars = ax.barh(feat_importance['feature'], feat_importance['coefficient'],
               color=colors, height=0.6, edgecolor='white')
ax.axvline(x=0, color='black', linewidth=0.8)
ax.set_title('Top 12 Features Driving Churn (Model Coefficients)', fontsize=13, pad=12)
ax.set_xlabel('Coefficient (positive = higher churn risk)')
ax.invert_yaxis()

plt.tight_layout()
plt.savefig('../images/09_feature_importance.png', bbox_inches='tight')
plt.show()
print('Chart saved: images/09_feature_importance.png')

## Step 10: Export Scored Dataset for Power BI

In [ ]:
# Export final scored dataset
# This is the file you load into Power BI
df_orig.to_csv('../data/telco_churn_scored.csv', index=False)

print('Scored dataset exported to: data/telco_churn_scored.csv')
print(f'Shape: {df_orig.shape}')
print('\nColumns added:')
print(' - churn_probability: 0–100 score for each customer')
print(' - risk_tier: High Risk / Medium Risk / Low Risk')
print('\nLoad this file into Power BI Desktop to build the dashboard.')

---
## ✅ Notebook 3 Complete

**What we built:**
- Logistic Regression model trained on 80% of 7,043 customers
- Model evaluated with Accuracy, Recall, F1, and ROC-AUC
- Scored all customers with a churn probability (0–100)
- Assigned High / Medium / Low risk tiers
- Exported `telco_churn_scored.csv` for Power BI

**Next:** Open Power BI Desktop and load `data/telco_churn_scored.csv`  
Then follow the Power BI build guide in `reports/powerbi_build_guide.md`